Dataset:
California Housing
Wymagania: <br>
Stwórz nowe cechy: interakcje (MedInc * AveRooms), transformacje (log, sqrt), bins <br>
Porównaj R² przed i po feature engineering <br>
Które nowe cechy najbardziej poprawiły model? <br>
Oczekiwany rezultat: (challenge) <br>
Lista nowych cech <br>
Porównanie R² (co najmniej 5% poprawa)<br>


In [6]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Dane
data = fetch_california_housing()

#Tworzymy tabele X z kolumnami - nazwami cech
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target #target to cena domu

#podział danych na test i train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model bazowy
model_base = LinearRegression() #pusty model
model_base.fit(X_train, y_train) #model liczy współczynniki metoda najmn. kwadratów

y_pred_base = model_base.predict(X_test) #model liczy przewidywania
r2_base = r2_score(y_test, y_pred_base) #liczymy jakość modelu

print("R2 bazowe:", r2_base)


R2 bazowe: 0.5757877060324512


In [7]:
#Model wyjaśnia 59% wariancji cen

#Nowa kolumna (cena w zależności od okolicy i powierzchni)
X["MedInc_AveRooms"] = X["MedInc"] * X["AveRooms"]

#logarytm (bez log model zakłada , że każdy dodatkowy mieszkaniec zwiększa cenę o tyle samo)
X["log_Population"] = np.log1p(X["Population"])

#logarytm (l. os w domu)
X["log_AveOccup"] = np.log1p(X["AveOccup"])

#Pierwiastek - model może dopasować krzywą która rośnie wolniej (dochód 1->2 to duza różnica, ale z 8->10 już nie - trzeba uchwycić "malejący efekt krańcowy")
X["sqrt_MedInc"] = np.sqrt(X["MedInc"])

#Binning (zmiana zakresu dochodu. na 4 przedziały: 0-3)
X["Income_bin"] = pd.cut(
    X["MedInc"],
    bins=4,
    labels=False
)

#Trening po FE
X_train_fe, X_test_fe, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model_fe = LinearRegression()
model_fe.fit(X_train_fe, y_train)
y_pred_fe = model_fe.predict(X_test_fe)
r2_fe = r2_score(y_test, y_pred_fe)
print("R2 po FE:", r2_fe)



R2 po FE: 0.6309576312380136


In [8]:
#Sprawdzanie ważności cech:
coefs = pd.Series(
    model_fe.coef_,
    index=X_train_fe.columns
).abs().sort_values(ascending=False)
coefs


log_AveOccup       1.376973
AveBedrms          0.544760
Latitude           0.424494
Longitude          0.418040
MedInc             0.324800
sqrt_MedInc        0.232868
Income_bin         0.092914
AveRooms           0.086171
log_Population     0.028538
HouseAge           0.011229
AveOccup           0.006602
MedInc_AveRooms    0.002414
Population         0.000026
dtype: float64

Najbardziej poprawiły model:

log_AveOccup – największy wpływ (|coef| = 1.37)

sqrt_MedInc – istotny wpływ (|coef| = 0.23)

Income_bin – umiarkowany wpływ (|coef| = 0.09)
